# Notebook 2: Digital twins with on-the-fly calibrated predictions


Welcome to notebook 2. In notebook 1, we investigated the effects of digital-twin-aided autoscaling on ICT infrastructure, with OpenDT coupled with an in-silico SURF cluster. We quantified OpenDT's prediction on power draw within 10% error rate, yet below the ground truth almost all the time (we call this underestimation).

Like many state-of-the-art simulators for datacenters, OpenDC uses a deterministic and static simulation model, while hardware behavior varies with temperature, aging, versions of the software stack, and diverse unobserved (but occurring) factors. Exhaustive datacenter modelling is computationally and conceptually unfeasible, and simulators often rely on more simplistic and "good enough" predictive approaches, with an acceptable error rate. Digital twins can help alleviate this error rate, with calibration. In this notebook, focused on simulation calibration, you will focus on the energy model and will learn:

1. How OpenDC the models the power draw of a host, and what the calibration parameter *r* does;
2. How a 1D grid search (i.e., sweep) over *r* finds the most accurate model, based on the last window of operation;
3. How OpenDT's self-calibrator keeps the twin accurate, with a configurable past-lookup period;
4. Where are the trade-offs: how much can we search for "the optimal" *r*, before the search becomes unfeasible.

By the end of the notebook, you will be able to explain why a digital twin needs calibration, how to run calibration yourself, and reason about suitable recommendation parameters by analyzing trade-offs.


_For the best learning experience, we strongly recommend starting with notebook 1._

![Self-calibration in OpenDT](resources/images/calibration.gif)

*Self-calibration: when the simulation drifts from the telemetry, OpenDT calibrates (red C); when they match, r stays as it is (green C).*

> Run the cells in order (Shift+Enter). For each question, click your selected option to reveal the answer.

In [ ]:
import numpy as np
import pandas as pd
from resources import *

check_setup()

# 1. How does OpenDC simulate power draw?

OpenDC simulates (instantaneous) power draw, via one of the 8 supported power models, out of which two support calibration. OpenDT uses OpenDC's MSE power model:

$P(u) = P_{idle} + (P_{max} - P_{idle})(2u - u^r)$.
- $u$ is device utilization,
- $P_{idle}$ and $P_{max}$ represent idle and max power,
- and $r$ is the calibration parameter.

For the SURF hosts, OpenDT uses $P_{idle}$ = 25 W, $P_{max}$ = 174 W, and the starting value of $r$ = 4; remember, **r** is the parameter OpenDT changes to calibrate predictions.

In [ ]:
u = np.linspace(0, 1, 101)  # CPU utilization, from 0 (idle) to 1 (full load)
P_IDLE, P_MAX = 25, 174  # watts
r_vals = [2,4,6,9]


curves = {f"r = {r}": P_IDLE + (P_MAX - P_IDLE) * (2 * u - u**r) for r in r_vals}
ax = pd.DataFrame(curves, index=u * 100).plot(figsize=(4,2))
ax.set(xlabel="CPU Utilization [%]", ylabel="Power Draw [W]", xlim=(0, 100), ylim=(0, None))

legend_above(ax)

In [ ]:
ask_question("2.1")

In [ ]:
ask_question("2.2")

# 2. Calibrating "r" with a grid-search

The Self-Calibrator emplyos a grid search strategy over the power model's parameter space. The current version of OpenDT supports one-dimensional grid search exploration, which is also called a "sweep". We now detail how a digital twin can calibrate its power model, with the example of OpenDT.

**Step 0:** understand the power model $P(u) = P_{idle} + (P_{max} - P_{idle})(2u - u^r)$.

**Step 1:** identify the calibration factor -- N factors=N-dimensional grid, 1 factor=1D grid. For us, it's **r**.

**Step 2:** give a starting value e.g., r=4, and exploration boundary e.g., explore r between [3, 9]

**Step 3:** decide how many exploration calibration searches to make e.g., 7 searches between [3, 9], equally split (3, 4, 5, ..., 9)

**Step 4:** Quantify error rates per simulation.

**Step 5:** Select **r** which has led to the lowest MAPE.

In [ ]:
R_VALUES = [3, 4, 5, 6, 7, 8, 9]
sweep = simulate_many("r", R_VALUES, until=TRACE_END)  # one simulation of the full week per r

def error(run):
    """Error of a simulation at each minute [%]: |ground truth - prediction| / ground truth."""
    both = align(run)  # ground truth and prediction on the same 1-minute grid
    return (both["measured"] - both["simulated"]).abs() / both["measured"] * 100

errors = pd.DataFrame({r: error(run) for r, run in sweep.items()})  # one column per r
pd.DataFrame([errors.mean()], index=["MAPE [%]"]).rename_axis(columns="r").style.format(precision=1)

In [ ]:
ask_question("2.3")

We now look at steps 4 and 5, window by window, from 9:00 to 17:00 UTC on 7 October, each window of 1 hour. Each window's best r (in bold) is the **r** OpenDT uses to simulate the next window.


In [ ]:
hourly = errors.resample("1h").mean()  # Step 4, per one-hour window: the MAPE of each r
best_r = hourly.idxmin(axis=1)  # Step 5: the best r of each window...
used_r = best_r.shift(1, fill_value=4)  # ...is the r OpenDT simulates the next window with

hours = slice("2022-10-07 09:00", "2022-10-07 16:00")  # eight windows, 9:00 to 17:00 UTC on 7 October
example = hourly.loc[hours].copy()
example["Best r"] = best_r.loc[hours]
example["OpenDT used r"] = used_r.loc[hours]
example.index = [f"{h:%H:%M}-{h + pd.Timedelta('1h'):%H:%M}" for h in example.index]
example.style.format(precision=1, subset=R_VALUES).highlight_min(axis=1, subset=R_VALUES, props="font-weight: bold")

In [ ]:
ask_question("2.4")

In [ ]:
with_calibration = pd.Series([hourly.loc[h, r] for h, r in used_r.items()], index=hourly.index)  # each hour, with the r used
for name, mape_per_hour in [("Self-calibrating every hour", with_calibration), ("No calibration (r = 4)", hourly[4])]:
    print(f"{name}: MAPE {mape_per_hour.mean():.1f}%, hours within 10%: {(mape_per_hour < 10).mean():.0%}")

# 3. The Best r per Day

In [ ]:
best_r_per_day = errors.resample("24h", origin=TRACE_START).mean().idxmin(axis=1)
days = [f"Day {d}" for d in range(1, len(best_r_per_day) + 1)]
pd.DataFrame([best_r_per_day.values], index=["Best r"], columns=days)

In [ ]:
ask_question("2.5")

# 4. Trade-offs

When resources are limited, calibration can be costly. For example, when deploying OpenDT on a laptop with just a few cores, running 100s of simulations (equivalent to exploring 100s of values for r) can be unfeasible. Also, if the digital twin is fed telemetry and needs to make decisions, either autonomously or with a human in the loop, every x minutes, the total time for simulation, calibration, and decisions needs to be under x minutes. We present two trade-offs, and you will bring two more (use the hints below, if needed).

#### 4.1 Trade-off between breadth and depth, in the grid
Each value of r in the grid is one more simulation per window. With a fixed budget (e.g., 5 simulations per window), the grid can either cover a wide range of r with coarse steps (breadth, e.g., r = 1, 10, 20, 30, 40), or a narrow range with fine steps (depth, e.g., r = 5, 6, 7, 8, 9). The next cell compares both grids.

In [ ]:
GRIDS = {"Depth: 5,6,7,8,9": [5, 6, 7, 8, 9], "Breadth: 1, 10, 20, 30, 40": [1, 10, 20, 30, 40]}
compare_grids(GRIDS)  # replays the Self-Calibrator with each grid (code in resources/calibration.py)

In [ ]:
ask_question("2.6")

#### 4.2 Trade-off between accuracy and cost, in the frequency of calibration

So far, OpenDT recalibrates after every window of one hour, with 7 simulations each time (i.e., 168 simulations per day). However, OpenDT simulates from the start of the trace at every window, so each simulation takes longer with every day the twin runs (e.g., on one core, 2.6 s for the first day of SURF, and 4.2 s for the whole week). To save simulations, the digital twin can recalibrate less often (e.g., every 6 hours), and keep the same r in between. The next cell replays the Self-Calibrator from every hour to every 48 hours.

In [ ]:
compare_frequencies(R_VALUES, every=[1, 2, 6, 12, 24, 48])  # hours between two calibrations (code in resources/calibration.py)

In [ ]:
ask_question("2.7")

#### 4.3 Trade-off between time and x (your turn: fill in x)

Each calibration takes time, and the digital twin needs to make decisions before the next window starts.

Besides the values of r (section 4.1), which other choice makes each calibration take longer, but can make the twin more accurate?
Discuss it with your pair, and use the hint if needed.

<details><summary><b>Hint</b></summary>Step 1 in section 2: N calibration factors make an N-dimensional grid. OpenDT calibrates only r; what if it also calibrated P<sub>idle</sub> and P<sub>max</sub>?

<details><summary><b>Answer</b></summary> <b>adding multiple dimensions for calibration</b>. x is the number of calibration factors (i.e., the dimensions of the grid): with 7 values each, r alone needs 7 simulations per window, r and P<sub>idle</sub> need 49 (7 x 7), and r, P<sub>idle</sub>, and P<sub>max</sub> need 343 (7 x 7 x 7).</details>
</details>

#### 4.4 One more trade-off (your turn)
Can you identify another trade-off introduced by self-calibration?

*Next:* [3.co2_migration.ipynb](3.co2_migration.ipynb), CO2-aware workload migration.